# Volet B : effet du choix de modèle sur la valeur d'un portefeuille

Nelson-Siegel (modèle testé) contre Svensson (référence, reproduit la courbe BCE à moins de 0,05 pb). Portefeuille fictif : 4 obligations à coupon annuel de 3 %, nominal 100, maturités 2, 5, 10 et 20 ans. Prix = somme des flux × exp(−r(t)·t) (convention continue de la BCE). Les choix (portefeuille, chocs, mesures) sont fixés avant de regarder les résultats.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.curves.nelson_siegel import ns_rate
from src.curves.svensson import sv_rate
from src.curves.portefeuille import (flux_obligations, choc_parallele, choc_pente,
                                     erreurs_valorisation)

NOMS_NS = ["beta0", "beta1", "beta2", "tau"]
NOMS_SV = ["beta0", "beta1", "beta2", "beta3", "tau1", "tau2"]

ns = pd.read_csv("../data/processed/hist_ns.csv", index_col=0, parse_dates=True)
sv = pd.read_csv("../data/processed/hist_sv.csv", index_col=0, parse_dates=True)
assert ns.index.equals(sv.index)
print(sv["rmse_pb"].max(), (sv["rmse_pb"] > 0.05).sum())   # contrôle : nouveau fichier (≈ 0,048 pb et 0)

0.0481776284452059 0


In [2]:
# choix fixés avant les résultats
DATE_HAUSSE = pd.Timestamp("2022-07-27")
MATURITES = [2, 5, 10, 20]
COUPON = 0.03
TOLERANCE_PRIX = 0.001      # 0,1 % : même tolérance que pour les taux
SCENARIOS = {"parallèle +100 pb": choc_parallele(100),
             "pente 5 pb/an, pivot 10 ans": choc_pente(5, 10)}
COLS = [f"{m} ans" for m in MATURITES] + ["portefeuille"]

flux = flux_obligations(MATURITES, COUPON)

In [3]:
def resume(err, tol=None):
    """Résumé par régime en % : biais, médiane, q95 et max de |erreur| (+ part de |erreur| > tol)."""
    avant = np.asarray(err.index < DATE_HAUSSE)
    blocs = {}
    for nom, m in [("avant", avant), ("après", ~avant)]:
        e = err[m]
        t = pd.DataFrame({"biais": e.mean(), "médiane |err|": e.abs().median(),
                          "q95 |err|": e.abs().quantile(0.95), "max |err|": e.abs().max()}) * 100
        if tol is not None:
            t["part > tol"] = (e.abs() > tol).mean()
        blocs[nom] = t
    return pd.concat(blocs, axis=1)

# 1) erreur de valeur (courbe inchangée) : ne dépend pas du choc
valeur, _ = erreurs_valorisation(sv, sv_rate, NOMS_SV, ns, ns_rate, NOMS_NS, flux, SCENARIOS["parallèle +100 pb"])
valeur.columns = COLS
print("Erreur de valeur NS - référence (en % du prix)")
print(resume(valeur, TOLERANCE_PRIX).round(3))

Erreur de valeur NS - référence (en % du prix)
              avant                                               après  \
              biais médiane |err| q95 |err| max |err| part > tol  biais   
2 ans         0.069         0.070     0.220     0.421      0.373  0.098   
5 ans        -0.136         0.068     0.609     0.930      0.349 -0.164   
10 ans       -0.005         0.035     0.359     0.938      0.296 -0.075   
20 ans        0.282         0.057     1.393     1.815      0.359  0.495   
portefeuille  0.051         0.016     0.238     0.303      0.237  0.088   

                                                           
             médiane |err| q95 |err| max |err| part > tol  
2 ans                0.088     0.234     0.413      0.388  
5 ans                0.156     0.308     0.648      0.769  
10 ans               0.101     0.234     0.361      0.504  
20 ans               0.525     0.821     0.984      0.920  
portefeuille         0.098     0.149     0.177      0.480  


In [8]:
#1') on test plusieurs tolérances, que le 0.1% précédent
TOLS = [0.001, 0.0025, 0.005, 0.01]      # 0,1 % (tolérance principale), 0,25 %, 0,5 %, 1 %
avant = np.asarray(valeur.index < DATE_HAUSSE)
out = {tol: pd.concat({"avant": (valeur[avant].abs() > tol).mean(),
                       "après": (valeur[~avant].abs() > tol).mean()}, axis=1)
       for tol in TOLS}
print(pd.concat(out, axis=1, names=["tolérance", "régime"]).round(3))

tolérance    0.0010        0.0025        0.0050        0.0100      
régime        avant  après  avant  après  avant  après  avant après
2 ans         0.373  0.388  0.012  0.039  0.000  0.000  0.000   0.0
5 ans         0.349  0.769  0.209  0.159  0.104  0.005  0.000   0.0
10 ans        0.296  0.504  0.129  0.032  0.016  0.000  0.000   0.0
20 ans        0.359  0.920  0.287  0.855  0.230  0.540  0.131   0.0
portefeuille  0.237  0.480  0.033  0.000  0.000  0.000  0.000   0.0


In [4]:
# 2) erreur de sensibilité sous chaque choc
for nom, choc in SCENARIOS.items():
    _, sens = erreurs_valorisation(sv, sv_rate, NOMS_SV, ns, ns_rate, NOMS_NS, flux, choc)
    sens.columns = COLS
    print(f"\nErreur de sensibilité, choc {nom} (en points de % du prix)")
    print(resume(sens).round(3))


Erreur de sensibilité, choc parallèle +100 pb (en points de % du prix)
              avant                                    après                \
              biais médiane |err| q95 |err| max |err|  biais médiane |err|   
2 ans        -0.000         0.000     0.000     0.000 -0.000         0.000   
5 ans         0.000         0.000     0.002     0.002  0.001         0.001   
10 ans       -0.000         0.000     0.006     0.014  0.000         0.001   
20 ans       -0.011         0.002     0.056     0.074 -0.020         0.021   
portefeuille -0.007         0.001     0.037     0.046 -0.013         0.014   

                                  
             q95 |err| max |err|  
2 ans            0.000     0.000  
5 ans            0.001     0.002  
10 ans           0.003     0.005  
20 ans           0.033     0.040  
portefeuille     0.022     0.025  

Erreur de sensibilité, choc pente 5 pb/an, pivot 10 ans (en points de % du prix)
              avant                                   